## Simple Example

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.messages import HumanMessage
from langchain_ollama import OllamaEmbeddings
from mineru import MinerU
from langchain_mineru import MinerULoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, MarkdownTextSplitter,  MarkdownHeaderTextSplitter
import os
import json
import bm25s
from typing import List, Tuple, Dict
import jieba as seg
from pymilvus import MilvusClient
from pymilvus import Function, FunctionType, DataType


from dotenv import load_dotenv
load_dotenv(override=True)

qwen_model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))
mineru_pdf_client  = MinerU(token=os.getenv("MINERU_TOKEN"))
embedding_model = "qwen3.7-text-embedding"

resources_dir = "/home/anthony/Learning/Projects/awesome-agentic-ai-zh/my_notebooks/resources"
maotai_pdf_fp = "/home/anthony/Learning/Projects/awesome-agentic-ai-zh/my_notebooks/resources/贵州茅台研报.pdf"
corpus_md_fp = "/home/anthony/Learning/Projects/awesome-agentic-ai-zh/my_notebooks/resources/中二知识笔记.md"

/tmp/ipykernel_247625/3132935599.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/anthony/Learning/Projects/awesome-agentic-ai-zh/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ===============一、构建知识库阶段====================
# =============1.加载文档===============
loader = PyPDFLoader(
    "resources/贵州茅台研报.pdf",
    mode="single", # single \ page
)
docs = loader.load()
print(f"文档数量:{len(docs)}")

# =============2.切分文档===============
# 创建递归切分器
splitter = CharacterTextSplitter(separator="\n", chunk_size=1200, chunk_overlap=150)
# 切分文档
chunks = splitter.split_documents(docs)
print(f"分块数量：{len(chunks)}")

# =============3.向量化===============
# 向量模型，这里用阿里云的向量模型
embeddings = DashScopeEmbeddings(
    model="qwen3.7-text-embedding",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 创建向量库，需要指定向量模型，存储文档时会自动调用向量模型完成向量化
vectorstore = InMemoryVectorStore(embedding=embeddings)
# 添加文档
vectorstore.add_documents(chunks)


# ==============二、在线问答阶段================
def try_rag(query: str):
    # 1.检索文档（VectorStore会自动把问题向量化，召回相关知识片段）
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    # 2.拼接上下文提示词
    content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"""你基于我提供的报告回答用户问题，报告中没提及的就说不知道，不要自己编造答案.
    report: ```{content}```
    query: {query}"""
    print(f"prompt: {prompt}")
    print("--------------------------------")
    # 3.调用模型，生成答案
    response = qwen_model.invoke(prompt)
    return response.content

文档数量:1
分块数量：19


In [7]:
print(try_rag("请帮我分析一下贵州茅台2026上半年盈利情况"))

基于您提供的报告，贵州茅台2026年上半年的盈利情况分析如下：

**1. 利润增速转负**
*   **归母净利润**：实现445.17亿元，同比下降1.95%。
*   **扣非归母净利润**：实现444.64亿元，同比下降2.04%。

**2. 利润率同步承压**
*   **毛利率**：上半年毛利率为89.56%，同比下降1.74个百分点（其中飞天茅台和系列酒毛利率同比分别下降1.57pct和4pct）。
*   **净利率**：净利率为50.75%，同比下降1.81个百分点。

**3. 费用率情况**
*   **销售费用率**：为3.47%，同比下降0.10个百分点。
*   **管理费用率**：为4.06%，同比下降0.07个百分点。
*   报告指出，两费率再收窄空间有限，这也是净利率承压的因素之一。

**4. 盈利下滑/承压的主要原因**
*   茅台终端价格体系松动后，公司采取放量补价策略以迎合市场需求。
*   产品结构调整和渠道变革带来了吨价下行压力，直接拖累了利润的增长。


In [8]:
print(try_rag("茅台收盘价多少"))

根据报告提供的数据，茅台的收盘价为 1,257.12 元。


In [17]:
docs[0].__dict__

{'id': None,
 'metadata': {'producer': '; modified using iTextSharp 5.2.1 (c) 1T3XT BVBA',
  'creator': '',
  'creationdate': '',
  'author': '',
  'comments': '',
  'company': '',
  'keywords': '',
  'moddate': '2026-09-21T15:43:49+08:00',
  'sourcemodified': "D:20260918160437+08'00'",
  'subject': '',
  'title': '诚通证券-贵州茅台(600519)公司跟踪报告：以动销定投放，缓解市场压力，低速稳态发展-260918.pdf',
  'trapped': 'False',
  'source': 'resources/贵州茅台研报.pdf',
  'total_pages': 15},
 'page_content': '敬请参阅最后一页免责声明 -1- 证券研究报告\n2026 年 09 月 18 日\n贵州茅台（600519.SH） 食品饮料/白酒Ⅱ\n以动销定投放，缓解市场压力，低速稳态发展\n——贵州茅台公司跟踪报告 公司跟踪\n\uf06c 贵州茅台 2026 年上半年收入增速放缓，利润增速转负：\n公司 2026 年中报实现营收 922.78 亿元/+1.30%；归母净利润 445.17 亿元\n/-1.95%；扣非归母净利润 444.64 亿元/-2.04%。茅台终端价格体系松动后，\n公司放量补价迎合市场需求，产品结构调整和渠道变革带来的吨价下行压\n力，拖累利润增长，公司毛利率上半年 89.56%/-1.74pct。两费方面，销售费\n用率 3.47%/-0.10pct；管理费用率 4.06%/-0.07pct，净利率 50.75%/-1.81pct。\n两费率再收窄空间有限，净利率同步承压。\n二季度单季营收净利负增长，2026Q1/Q2 分别实现营收 54703/375.75 亿元，\n同 比 增 长 6.34%/-5.23% ； 归 母 净 利 润 272.43/172.74 亿 元 同 比 增 长\n1.47%//-6.90% ，

## Offline Knowleage Store
- File Extractor 
    - **Mineru**: free pdf extractor, ocr ok, free
- Doc Chunk
- Vector Store
    - embedding
    - retrieve

In [ ]:
# 解析文件
result = mineru_pdf_client.flash_extract(maotai_pdf_fp)

# 获取markdown
print(result.markdown)

In [ ]:
loader = MinerULoader(
    source=maotai_pdf_fp,
    mode="flash" # 可选: flash 、 precision
)
# 解析文档，返回值直接是LangChain的 Document集合
docs = loader.load()

print(docs[0].metadata) # 元数据
# print(docs[0].page_content) # 文档内容

# 写到本地看看
with open(f"{resources_dir}/r1.md", "w", encoding="utf-8") as f:
    f.write(docs[0].page_content)

{'source': '/home/anthony/Learning/Projects/awesome-agentic-ai-zh/my_notebooks/resources/贵州茅台研报.pdf', 'loader': 'mineru', 'output_format': 'markdown', 'mode': 'flash', 'language': 'ch', 'pages': None, 'split_pages': False, 'filename': None}


In [ ]:
doc = """
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。
"""

# 创建字符切分器
text_splitter = CharacterTextSplitter(
    separator="",        # 分隔符，空就是任意位置都可分隔
    chunk_size=200,     # 每块最大200字符
    chunk_overlap=40,   # 块之间重叠40字符
)


# 切分文本
chunks = text_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} 字符")
print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

原始文本长度: 706 字符
切分为 5 个块:

--- Chunk 1 (199字符) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等

--- Chunk 2 (200字符) ---
部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后

--- Chunk 3 (200字符) ---
强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。Recur

--- Chunk 4 (200字符) ---
xtSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

通过合理的实验设

--- Chunk 5 (65字符) ---
文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

通过合理的实验设计和评估，开发者

In [ ]:
# 创建字符切分器
# 使用from_tiktoken_encoder，LangChain自带，无需额外安装tiktoken
token_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",    # token分词器编码名
    chunk_size=200,                # 每块最多200 token
    chunk_overlap=40,              # 块之间重叠40 token
)

chunks = token_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} 字符")
print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

Created a chunk of size 337, which is longer than the specified 200


原始文本长度: 706 字符
切分为 4 个块:

--- Chunk 1 (123字符) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

--- Chunk 2 (319字符) ---
检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

--- Chunk 3 (223字符) ---
LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

--- Chunk 4 (33字符) ---
通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。



In [ ]:

# 创建递归切分器
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", "。"]  # 优先级从高到低
)

# 切分文档
chunks = recursive_splitter.split_text(doc)

print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

切分为 6 个块:

--- Chunk 1 (123字符) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

--- Chunk 2 (183字符) ---
检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性

--- Chunk 3 (170字符) ---
。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

--- Chunk 4 (177字符) ---
LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器

--- Chunk 5 (46字符) ---
。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

--- Chunk 6 (33字符) ---
通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。



In [ ]:
markdown_splitter = MarkdownTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    # separators=["\n\n", "\n", "。"]  # 优先级从高到低
)

# 切分文档
chunks = markdown_splitter.split_text(maotai_pdf_fp)

print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

切分为 6 个块:

--- Chunk 1 (123字符) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

--- Chunk 2 (199字符) ---
检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统

--- Chunk 3 (160字符) ---
模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

--- Chunk 4 (199字符) ---
LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略

--- Chunk 5 (64字符) ---
angChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

--- Chunk 6 (33字符) ---
通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。



In [13]:
# 创建Embedding模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 向量化单条文本
text = "我爱上班"
vector = ollama_embeddings.embed_query(text)


# 批量向量化
texts = ["我要躺平", "我爱工作", "拒绝加班"]
vectors = ollama_embeddings.embed_documents(texts)
print(f"\n批量向量化: {len(vectors)} 条, 维度: {len(vectors[0])}")


print('==============衡量相似度===============')
import numpy as np


def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 比较向量相似度
for i, v in enumerate(vectors):
    similarity = cosine_similarity(vector, v)
    print(f"{texts[i]}:", similarity)


批量向量化: 3 条, 维度: 1024
==============衡量相似度===============
我要躺平: 0.4134495462584254
我爱工作: 0.8729641608924325
拒绝加班: 0.42409579124240526


In [22]:
# 初始化DashScopeEmbeddings，只需要指定模型名，会自动取.env读取DASHSCOPE_API_KEY
embeddings = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)


# 向量化单条文本
text = "我爱上班"
# 向量化后返回的是 list[float]
vector = embeddings.embed_query(text)

print(f"文本: {text}")
print(f"向量维度: {len(vector)}")
print(f"向量前5维: {vector[:5]}")


# 批量向量化
texts = ["我要躺平", "我爱工作", "拒绝加班"]
# 批量向量化，可以理解成处理很多个知识片段，返回 list[list[float]]
vectors = embeddings.embed_documents(texts)

print(f"\n批量向量化: {len(vectors)} 条, 维度: {len(vectors[0])}")


print('==============衡量相似度===============')
import numpy as np


def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 比较向量相似度
for i, v in enumerate(vectors):
    similarity = cosine_similarity(vector, v)
    print(f"{texts[i]}:", similarity)

文本: 我爱上班
向量维度: 1024
向量前5维: [0.014019661583006382, 0.0022319781128317118, -0.046387284994125366, -0.0060034687630832195, 0.05380723252892494]

批量向量化: 3 条, 维度: 1024
==============衡量相似度===============
我要躺平: 0.39420110112124457
我爱工作: 0.8183995384373998
拒绝加班: 0.4788502095332923


In [32]:
# =============1.准备文档================
with open(f"{resources_dir}/r1.md", 'r', encoding='utf-8') as f:
    doc = f.read()

# =============2.切分文档================
# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 在正文中是否去除标题，False，不去除
)

# 切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(doc)

for i, doc in enumerate(docs, start=1):
    doc.id = f'doc_{i}'

In [34]:
# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)

In [35]:
ids = vector_store.add_documents(docs)

print(ids)

['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5', 'doc_6', 'doc_7', 'doc_8', 'doc_9', 'doc_10', 'doc_11', 'doc_12', 'doc_13', 'doc_14', 'doc_15', 'doc_16', 'doc_17', 'doc_18', 'doc_19', 'doc_20', 'doc_21', 'doc_22', 'doc_23', 'doc_24', 'doc_25', 'doc_26', 'doc_27', 'doc_28', 'doc_29', 'doc_30', 'doc_31', 'doc_32', 'doc_33', 'doc_34']


In [46]:
# 用户问题
query = "作者是谁"

# 相似度检索，返回值是list[Document]
results = vector_store.similarity_search(
    query=query,
    k = 2,
)

# 打印结果
for doc in results:
    print(doc.model_dump_json(indent=2))

{
  "id": "doc_6",
  "metadata": {
    "h1": "以动销定投放，缓解市场压力，低速稳态发展",
    "h2": "强烈推荐（维持评级）"
  },
  "page_content": "## 强烈推荐（维持评级）  \n陈文倩（分析师）  \nchenwenqian@cctgsc.com.cn  \n登记编号：S0280515080002",
  "type": "Document"
}
{
  "id": "doc_28",
  "metadata": {
    "h1": "以动销定投放，缓解市场压力，低速稳态发展",
    "h2": "分析师声明"
  },
  "page_content": "## 分析师声明  \n本报告署名分析师具有中国证券业协会授予的证券投资咨询执业资格并登记为注册分析师，认真审慎、专业严谨、独立客观的出具本报告并对报告内容和观点负责。  \n分析师的薪酬的任何组成部分无论是在过去、现在及将来均不会与本报告中具体的推荐意见或观点有直接或间接的联系。",
  "type": "Document"
}


## Online Knowledge Retrieval

In [50]:
## 2-steps RAG
# 0.定义动态提示词中间件
@dynamic_prompt
def rag_prompt(request: ModelRequest):
    # 1.拿到用户问题，也就是消息列表中的最后一条消息
    print("================User Message================")
    user_input = request.state['messages'][-1].text
    print(user_input, "\n")

    # 2.检索知识片段, list[Document], 拼接为字符串
    docs = vector_store.similarity_search(user_input, k=2)
    context = "\n\n".join(doc.page_content for doc in docs)
    print('=============== Context ====================')
    print(context, "\n")

    # 3.拼接新系统提示词
    return f'''
    你是一个教资考试专家，你根据上下文回答用户问题，不能自己编造答案。答案要简短，直接引用原文。
    Context: {context}
    '''
# 1.初始化模型

# 2.定义Agent
agent = create_agent(
    qwen_model,
    middleware=[rag_prompt],
)

response = agent.invoke({"messages":[HumanMessage("论语中教育的目的是什么？")]})
print("==============AI Message================")
print(response["messages"][-1].text)

================User Message================
论语中教育的目的是什么？ 

=============== Context ====================
## 免责声明  
诚通证券股份有限公司经中国证券监督管理委员会批复，已具备证券投资咨询业务资格。  
本报告由诚通证券股份有限公司（以下简称诚通证券）供其机构或个人客户（以下简称客户）使用，诚通证券无需因接收人收到本报告而视其为客户。本报告是发送给诚通证券客户的，属于机密材料，只有诚通证券客户才能参考或使用，如接收人并非诚通证券客户，请及时退回并删除。  
本报告所载的全部内容只供客户做参考之用，并不构成对客户的投资建议，并非作为买卖、认购证券或其它金融工具的邀请或保证。诚通证券根据公开资料或信息客观、公正地撰写本报告，但不保证该公开资料或信息内容的准确性或完整性。客户请勿将本报告视为投资决策的唯一依据而取代个人的独立判断。诚通证券不需要采取任何行动以确保本报告涉及的内容适合于客户。诚通证券建议客户如有任何疑问应当咨询证券投资顾问并独自进行投资判断。本报告并不构成投资、法律、会计或税务建议或担保任何内容适合客户，本报告不构成给予客户个人咨询建议。  
本报告所载内容反映的是诚通证券在发表本报告当日的判断，诚通证券可能发出其它与本报告所载内容不一致或有不同结论的报告，但诚通证券没有义务和责任去及时更新本报告涉及的内容并通知客户。诚通证券不对因客户使用本报告而导致的损失负任何责任。  
对于浏览过程中可能涉及的诚通证券网站以外的地址或超级链接，诚通证券不对其内容负责。链接网站的内容不构成本报告的任何部分，客户需自行承担浏览这些网站的费用或风险。  
除非另有说明，所有本报告的版权属于诚通证券。未经诚通证券事先书面授权，任何机构或个人不得以任何形式更改、复制、传播本报告中的任何材料，或以任何侵犯本公司版权的其他方式使用。所有在本报告中使用的商标、服务标识及标记，除非另有说明，均为诚通证券的商标、服务标识及标记。诚通证券版权所有并保留一切权利。

## 分析师声明  
本报告署名分析师具有中国证券业协会授予的证券投资咨询执业资格并登记为注册分析师，认真审慎、专业严谨、独立客观的出具本报告并对报告内容和观点负责。  
分析师的薪酬的任何组成部分无论是在过去、现在及将来

In [51]:
# 3.定义工具
@tool
def retrieve_docs(query: str):
    """检索知识库，获取与用户提问相关的知识片段"""

    # 1.根据问题检索知识片段
    documents = vector_store.similarity_search(query)

    # 2.拼接知识片段
    context = "\n".join([doc.page_content for doc in documents])

    return context

# 4.定义Agent，绑定model和Tool
agent = create_agent(
    qwen_model,
    tools=[retrieve_docs],
    system_prompt="你是一个股票研究员，回答与教资相关问题时，必须调用工具检索文档，根据文档回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。"
)

response = agent.invoke(
    {"messages": [HumanMessage("茅台该不该买入？")]} # 查询重写
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

茅台该不该买入？
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_d05c6ca8047b4b35844bfa2b)
 Call ID: call_d05c6ca8047b4b35844bfa2b
  Args:
    query: 茅台该不该买入
================================= Tool Message =================================
Name: retrieve_docs

# 以动销定投放，缓解市场压力，低速稳态发展  
## 贵州茅台公司跟踪报告
## 相关报告  
《顺势出清舒缓压力 巩固优势稳健前行贵州茅台 2025 年年报及 2026 年一季报点评》2026-05-22  
《茅台高端地位稳固，中端白酒竞争激烈——贵州茅台 2025 年中报点评》2025-08-14  
《2025 年稳健前行，双轮驱动优化产品结构— 贵州茅台 2024年年报点评》2025-04-14  
《主动控量助力企业稳健发展— 贵州茅台 2024 年三季报点评》2024-11-12
## 分析师声明  
本报告署名分析师具有中国证券业协会授予的证券投资咨询执业资格并登记为注册分析师，认真审慎、专业严谨、独立客观的出具本报告并对报告内容和观点负责。  
分析师的薪酬的任何组成部分无论是在过去、现在及将来均不会与本报告中具体的推荐意见或观点有直接或间接的联系。
## 5、 估值与投资建议  
白酒行业正经历着深刻的周期性与结构性调整，“存量竞争”和“马太效应”的特点也会愈发明显。行业承压之际，公司以消费者为中心、以市场需求为驱动，开启“全面向 C”的市场化改革；通过推动场景、客群、服务“三个转型”与产品端、渠道端、终端“三端变革”，有效稳住了市场基本盘，巩固飞天市场地位，展现企业发展韧性，缓解经销商压力，实现市场稳健发展。2026年茅台酒回归“金字塔型”产品结构，每一产

## RAG enhancement

In [53]:
# 读取markdown数据
docs = ""
with open(f"{resources_dir}/中二知识笔记.md", "r", encoding="utf-8") as f:
    lines = f.readlines()
    docs = "".join(line for line in lines)

# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

# 切分文档
chunks = markdown_splitter.split_text(docs)


idx = 1
# 定义方法，用来给文档内容拼上三级标题，让文档不丢失其所属章节，顺便加个id
def handle_doc_header(doc: Document):
    global idx
    m = doc.metadata
    doc.page_content = f"### {m['Header 3']}\n{doc.page_content}"
    doc.id = f"doc_{idx}"
    idx = idx + 1
    return doc
ds = [handle_doc_header(doc) for doc in chunks]

# 创建向量库和检索器
embeddings = DashScopeEmbeddings(
    model="qwen3.7-text-embedding", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)
vectorstore = InMemoryVectorStore(embeddings)

# 添加文档
vectorstore.add_documents(ds)

print(f"知识库已就绪，{len(chunks)} 个文档")

知识库已就绪，7 个文档


In [54]:
query = "哪个提出了要因材施教、启发诱导、学思结合的教学原则？"  # 发展区理论是谁搞出来的
retrieved_docs = vectorstore.similarity_search_with_score(query, k=3)
for doc, score in retrieved_docs:
    print(doc.model_dump_json(indent=2))
    print(f"=========score: {score}============")

{
  "id": "doc_3",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想\n- 教育作用：庶、富、教；性相近，习相远。\n- 教育对象：“有教无类”，教育民主思想。\n- 教育目的：以完善人格为教育的首要目的，培养士和君子。\n- 教学内容：文、行、忠、义。\n- 教学过程：学、思、习、行。\n- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。\n- 教师观：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
=========score: 0.6542309447897076============
{
  "id": "doc_5",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想\n- 人性论：性恶论，化性起伪，善德是后天习得的，重视教育的作用。\n- 教学原则：学以致用，锲而不舍。",
  "type": "Document"
}
=========score: 0.6333731046417416============
{
  "id": "doc_4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想\n- 人性论：人性本善，人先天具有仁、义、礼、智四个“善端”。\n- 教育作用：发扬善端，培养道德完人，得天下英才而教育之。\n- 教学原则：循序渐进，专心有恒。",
  "type": "Document"
}
=========score: 0.6262692313

#### Query Rewrite

In [55]:
rewrite_prompt = f"""
将以下问题改写为适合检索的关键词形式。提取出核心概念，用空格分隔。只输出关键词不要解释。

问题: {query}
关键词:
"""
response = qwen_model.invoke(rewrite_prompt)
rewrite_query = response.content
print(f"问题重写：'{query}' -> '{rewrite_query}'")
retrieved_docs = vectorstore.similarity_search_with_score(rewrite_query, k=3)
for doc, score in retrieved_docs:
    print(doc.model_dump_json(indent=2))
    print(f"=========score: {score}============")

问题重写：'哪个提出了要因材施教、启发诱导、学思结合的教学原则？' -> '因材施教 启发诱导 学思结合 教学原则 提出者'
{
  "id": "doc_2",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（二）近代西方教育家"
  },
  "page_content": "### （二）近代西方教育家\n- 夸美纽斯：提出班级授课制，著有《大教学论》，主张“把一切事物教给一切人”。\n- 卢梭：自然主义教育，强调儿童天性，著有《爱弥儿》。\n- 杜威：实用主义教育，主张“从做中学”，教育即生活、教育即生长。",
  "type": "Document"
}
=========score: 0.5909325908100271============
{
  "id": "doc_3",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想\n- 教育作用：庶、富、教；性相近，习相远。\n- 教育对象：“有教无类”，教育民主思想。\n- 教育目的：以完善人格为教育的首要目的，培养士和君子。\n- 教学内容：文、行、忠、义。\n- 教学过程：学、思、习、行。\n- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。\n- 教师观：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
=========score: 0.5900324252001274============
{
  "id": "doc_5",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想\n- 人性论：性恶论，化性起伪，善德

#### HyDE: Hypothesis Document Embedding

In [56]:
rewrite_prompt = f"""
请根据你的知识，生成一个对以下问题的可能答案（简短但要关键，50字左右可）：

问题: {query}
答案:
"""
response = qwen_model.invoke(rewrite_prompt)
fake_answer = response.content
print(f"虚构答案：'{query}' -> '{fake_answer}'")
retrieved_docs = vectorstore.similarity_search_with_score(fake_answer, k=2)
for doc, score in retrieved_docs:
    print(doc.model_dump_json(indent=2))
    print(f"=========score: {score}============")

虚构答案：'哪个提出了要因材施教、启发诱导、学思结合的教学原则？' -> '孔子。他在《论语》中提出了因材施教、启发诱导（不愤不启，不悱不发）和学思结合（学而不思则罔）等核心教学原则。'
{
  "id": "doc_3",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想\n- 教育作用：庶、富、教；性相近，习相远。\n- 教育对象：“有教无类”，教育民主思想。\n- 教育目的：以完善人格为教育的首要目的，培养士和君子。\n- 教学内容：文、行、忠、义。\n- 教学过程：学、思、习、行。\n- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。\n- 教师观：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
=========score: 0.7726413858180263============
{
  "id": "doc_5",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想\n- 人性论：性恶论，化性起伪，善德是后天习得的，重视教育的作用。\n- 教学原则：学以致用，锲而不舍。",
  "type": "Document"
}
=========score: 0.6146390713898227============


#### Mutiple hop

In [64]:
query = "孔子和孟子的教育思想有什么不同？"

rewrite_prompt = f"""
你是一个RAG专家，请将用户问题拆分为多个子问题，问题数量适当，不要过分拆解导致过渡检索引入噪音。不要任何解释，直接返回JSON数组。
问题: {query}
子问题:
"""
response = qwen_model.invoke(rewrite_prompt)
sub_queries = json.loads(response.content)
print(f"问题重写：'{query}' -> '{sub_queries}'")

retrieved_docs = {}

# 合并结果
for sub_query in sub_queries:
    docs = vectorstore.similarity_search_with_score(sub_query, k=2)
    retrieved_docs.setdefault(sub_query, docs)

# 滤重

for q, docs in retrieved_docs.items():
    print(f"*****************{q}******************")
    for i, d in enumerate(docs):
        print(f"==========排名:{i}===========")
        print(d[0].model_dump_json(indent=2))

问题重写：'孔子和孟子的教育思想有什么不同？' -> '['孔子的教育思想核心内容和主张是什么？', '孟子的教育思想核心内容和主张是什么？', '孔子和孟子在教育思想上的主要区别是什么？']'
*****************孔子的教育思想核心内容和主张是什么？******************
==========排名:0===========
{
  "id": "doc_3",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想\n- 教育作用：庶、富、教；性相近，习相远。\n- 教育对象：“有教无类”，教育民主思想。\n- 教育目的：以完善人格为教育的首要目的，培养士和君子。\n- 教学内容：文、行、忠、义。\n- 教学过程：学、思、习、行。\n- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。\n- 教师观：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
==========排名:1===========
{
  "id": "doc_4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想\n- 人性论：人性本善，人先天具有仁、义、礼、智四个“善端”。\n- 教育作用：发扬善端，培养道德完人，得天下英才而教育之。\n- 教学原则：循序渐进，专心有恒。",
  "type": "Document"
}
*****************孟子的教育思想核心内容和主张是什么？******************
==========排名:0===========
{
  "id": "doc_4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "

#### Dense Retrieval Vs Sparse Retrieval

#### Milvus

In [15]:


def prepare_docs():
    # =============1.准备文档================
    with open(corpus_md_fp, 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

docs = prepare_docs()

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)


In [83]:
docs[0].__dict__

{'id': None,
 'metadata': {'h1': '第一章 教育概述', 'h2': '第一节 中外教育家及其教育思想', 'h3': '（一）古代希腊教育家'},
 'page_content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）古代希腊教育家\n- 苏格拉底：产婆术，通过问答引导学生自己发现真理。\n- 柏拉图：主张教育应培养哲学王，强调理性与德性的统一。\n- 亚里士多德：提出德智体和谐发展，重视习惯养成与自由教育。',
 'type': 'Document'}

In [10]:
client = MilvusClient("http://localhost:19530")
collection_name = "my_collection_1"
if client.has_collection(collection_name=collection_name):
    client.drop_collection(collection_name=collection_name)

def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加BM25函数，可以自动把原始文档转BM25向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)

    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="INVERTED")  # 普通字段，采用哈希索引

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )

create_collection()

In [ ]:
res = client.list_collections()

print(res)  # ['my_collection_1']

['my_collection_1']


I1011 17:24:27.262122  247782 chttp2_transport.cc:1391] ipv4:127.0.0.1:46515: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1011 17:24:27.262287  247782 chttp2_transport.cc:1423] ipv4:127.0.0.1:46515: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [16]:
vectors = embeddings.embed_documents([doc.page_content for doc in docs])

In [20]:
data = [
    {
        "id": i+1,
        "content": doc.page_content,
        "h2": doc.metadata.get("h2", ""),
        "dense": vectors[i],  # 稠密向量字段
    }
    for i, doc in enumerate(docs)
]
# 插入
res = client.insert(collection_name, data)
print(res)

{'insert_count': 7, 'ids': [1, 2, 3, 4, 5, 6, 7]}


In [21]:
# 1.准备两条数据

# id为1的数据，修改其h2字段值，模拟更新
data1 = data[0]
data1['h2'] = "测试一下"

 # id为2的数据，将其id修改为88，模拟新增
data2 = data[1]
data2['id'] = 88

# 2.测试upsert
res = client.upsert(collection_name, [data1, data2])
print(res)# {'upsert_count': 2, 'ids': [1, 88]}

{'upsert_count': 2, 'ids': [1, 88]}


In [22]:
# 根据id删除
res = client.delete(collection_name, ids=[88])
print(res) # {'delete_count': 1, 'cost': 0}

{'delete_count': 1}
